# Sandbox SDK Quickstart

This notebook demonstrates the Python Sandbox SDK on prokube. It creates or uses a WarmPool, claims a sandbox, runs stateful Python code, executes shell commands, writes and reads files, and cleans up the sandbox.

The goal is to show the SDK and the core Agent Sandbox features. A separate example can show how to plug the same operations into an agent framework.

## Prerequisites

You need a prokube workspace with the Sandbox module enabled.

In a managed Lab, this notebook uses the in-cluster Agent Gateway service. For external access, set `PROKUBE_API_URL`, `PROKUBE_WORKSPACE`, and `PROKUBE_API_KEY` before using the SDK.

Do not paste real API keys into this notebook. Use environment variables or your notebook environment's secret handling.

## 1. Install the SDK

Run this cell if the SDK is not already installed in your notebook environment.

In [ ]:
%pip install -q "git+https://github.com/prokube/prokube-sdk.git@v0.1.3"

## 2. Configure the Client

Managed Labs can use the in-cluster Agent Gateway service directly. The cell below sets the SDK URL and derives the namespace from the mounted service account.

In [ ]:
import os
from pathlib import Path

workspace = Path("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read_text().strip()

os.environ.setdefault(
    "PROKUBE_API_URL",
    "http://agentgateway-proxy.agentgateway-system.svc.cluster.local",
)
os.environ.setdefault("PROKUBE_WORKSPACE", workspace)
os.environ.setdefault("PROKUBE_USER_ID", workspace)

sandbox_pool = os.environ.get("SANDBOX_POOL", "sandbox-sdk-quickstart")
sandbox_image = os.environ.get(
    "SANDBOX_IMAGE",
    "europe-west3-docker.pkg.dev/prokube-internal/prokube-customer/pk-sandbox-base:v14-05-2026",
)

print("Workspace:", os.environ["PROKUBE_WORKSPACE"])
print("API URL:", os.environ["PROKUBE_API_URL"])
print("WarmPool:", sandbox_pool)
print("Sandbox image:", sandbox_image)

### Optional: External Access

If you run this notebook outside the cluster, create an API key with Sandbox API access and set the SDK configuration explicitly. Do not store real API keys in the notebook.

In [ ]:
# Uncomment and fill these values when running outside the cluster.
# os.environ["PROKUBE_API_URL"] = "https://<cluster-domain>/pkui"
# os.environ["PROKUBE_WORKSPACE"] = "<workspace>"
# os.environ["PROKUBE_API_KEY"] = "<api-key>"

## 3. Create a WarmPool

WarmPools keep ready-to-claim sandboxes available for low-latency starts. This cell creates a small pool for the quickstart. If a pool with the same name already exists, choose a different `SANDBOX_POOL` value or delete the old pool first.

If your deployment uses a different sandbox image, set `SANDBOX_IMAGE` before running the notebook.

In [ ]:
import time

from prokube.sandbox import SandboxPool

pool = SandboxPool.create(
    name=sandbox_pool,
    image=sandbox_image,
    pool_size=1,
    cpu="1",
    memory="2Gi",
)
print(f"Created pool {pool.name}: {pool.ready_replicas}/{pool.pool_size} ready")

for _ in range(60):
    pool.refresh()
    if pool.ready_replicas > 0:
        break
    time.sleep(2)

print(f"Pool readiness: {pool.ready_replicas}/{pool.pool_size}")

## 4. Claim a Sandbox

Claiming reserves one ready sandbox from the WarmPool. Always clean up the sandbox when the task is done.

In [ ]:
from prokube.sandbox import Sandbox

sbx = Sandbox.from_pool(sandbox_pool)
print("Claimed sandbox:", sbx.name)
print("Initial status:", sbx.status)

## 5. Run Stateful Python Code

`run_code()` uses a stateful Python kernel. Imports and variables persist across calls while the sandbox is running.

In [ ]:
sbx.run_code("import statistics")
sbx.run_code("values = [1, 2, 3, 4, 5]")
result = sbx.run_code("print(statistics.mean(values))")
print(result.stdout)

## 6. Run Shell Commands

Use `commands.run()` for command-line tools inside the sandbox.

In [ ]:
command = sbx.commands.run("python --version")
print("exit code:", command.exit_code)
print(command.stdout)
print(command.stderr)

## 7. Work with Files

Files under `/workspace` are a good default for task data that should survive pause/resume.

In [ ]:
csv_data = "name,score\nalice,10\nbob,12\n"
sbx.files.write("/workspace/scores.csv", csv_data)

content = sbx.files.read("/workspace/scores.csv")
print(content.decode() if isinstance(content, bytes) else content)

files = sbx.files.list("/workspace")
for file_info in files:
    print(file_info)

## 8. Clean Up

Delete the sandbox when you are done. For production code, use `try`/`finally` or the SDK context manager so cleanup still runs after errors.

In [ ]:
sbx.kill()
print("Deleted sandbox:", sbx.name)

## Recommended Pattern

For scripts and agents, prefer a context manager so sandbox cleanup is automatic.

In [ ]:
with Sandbox.from_pool(sandbox_pool) as sandbox:
    result = sandbox.run_code("print('hello from a managed sandbox')")
    print(result.stdout)

## Optional Pool Cleanup

Delete the WarmPool if you created it only for this quickstart. Skip this cell if other users or jobs depend on the pool.

In [ ]:
# pool.delete()
# print("Deleted pool:", pool.name)